# Black Bird APK patcher for Google Colab

Installs tools, configure the target server, upload an APK, patch it for a private server, rebuild, sign, and download the patched APK.

**Patches**
- `global-metadata.dat` — rewrite IL2CPP string literals (URLs and hostnames)
- `libil2cpp.so` — ARM64 binary patches (SSL bypass, encryption passthrough, plain Octo list)
- `AndroidManifest.xml` — add `networkSecurityConfig` for cleartext HTTP (when using HTTP)
- `res/xml/network_security_config.xml` — allow cleartext traffic

**Quick Instructions**
- Configure `protocol`, `server_host`, and optional `server_port` below.
- Run the single code cell. It installs tools, patches, and downloads the result.

**Notes**
- Choose `http` to enable cleartext network patches; choose `https` to skip manifest/network changes.
- Replacements must not be longer than the original strings; use short hostnames or omit the port if needed.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
protocol = 'http'  #@param ['http', 'https']
server_host = '192.168.1.1'  #@param {type:'string'}
server_port = ''  #@param {type:'string'}

# ── Imports & constants ───────────────────────────────────────────────────────
import os
import re
import shutil
import stat
import struct
import subprocess
import urllib.parse
from dataclasses import dataclass
from pathlib import Path

WORK_ROOT = Path('/content/nier_patch_work')
TOOLS_DIR = WORK_ROOT / 'tools'
ANDROID_SDK_ROOT = TOOLS_DIR / 'android-sdk'
APKTOOL_VERSION = '3.0.1'
ANDROID_BUILD_TOOLS = '36.1.0'
ANDROID_CMDLINE_TOOLS_URL = 'https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip'

METADATA_MAGIC = 0xFAB11BAF
HDR_STRING_LITERAL_OFF = 8
HDR_STRING_LITERAL_DATA_OFF = 16

MOV_X0_0 = struct.pack('<I', 0xD2800000)
MOV_X0_X1 = struct.pack('<I', 0xAA0103E0)
RET = struct.pack('<I', 0xD65F03C0)
YEAR_9999 = struct.pack('<I', 0x5284E1E1)

IL2CPP_PATCHES = [
    {
        'name': 'ToNativeCredentials',
        'desc': 'SSL bypass — return NULL to force insecure gRPC channel',
        'rva': 0x35C8670,
        'bytes': MOV_X0_0 + RET,
    },
    {
        'name': 'HandleNet.Encrypt',
        'desc': 'encryption passthrough — return payload as-is',
        'rva': 0x279410C,
        'bytes': MOV_X0_X1 + RET,
    },
    {
        'name': 'HandleNet.Decrypt',
        'desc': 'decryption passthrough — return receivedMessage as-is',
        'rva': 0x279420C,
        'bytes': MOV_X0_X1 + RET,
    },
    {
        'name': 'OctoManager.Internal.GetListAes',
        'desc': 'Octo list: force plain list (return false = no AES); server serves raw list.bin',
        'rva': 0x4C27038,
        'bytes': MOV_X0_0 + RET,
    },
    {
        'name': 'TitleScreen.ForceYear9999',
        'desc': 'Replace EoS year 2024 with 9999',
        'rva': 0x2F11E7C,
        'bytes': YEAR_9999
    },
]

# ── Helper functions ──────────────────────────────────────────────────────────
def run(cmd, check=True, shell=False, env=None, cwd=None):
    printable = cmd if isinstance(cmd, str) else ' '.join(str(part) for part in cmd)
    print(f'$ {printable}')
    return subprocess.run(cmd, check=check, text=True, shell=shell, env=env, cwd=cwd)

def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path

# ── Tool installation ─────────────────────────────────────────────────────────
def install_apktool():
    ensure_dir(TOOLS_DIR)
    jar_path = TOOLS_DIR / f'apktool_{APKTOOL_VERSION}.jar'
    wrapper_path = TOOLS_DIR / 'apktool'

    if not jar_path.exists():
        run([
            'wget', '-q', '-O', str(jar_path),
            f'https://github.com/iBotPeaches/Apktool/releases/download/v{APKTOOL_VERSION}/apktool_{APKTOOL_VERSION}.jar',
        ])

    wrapper_path.write_text(
        '#!/usr/bin/env bash\n'
        f'exec java -jar "{jar_path}" "$@"\n',
        encoding='utf-8',
    )
    wrapper_path.chmod(wrapper_path.stat().st_mode | stat.S_IEXEC)
    os.environ['PATH'] = f'{TOOLS_DIR}:{os.environ["PATH"]}'

def install_android_commandline_tools() -> Path:
    ensure_dir(TOOLS_DIR)
    ensure_dir(ANDROID_SDK_ROOT)

    zip_path = TOOLS_DIR / 'commandlinetools.zip'
    temp_dir = TOOLS_DIR / 'cmdline-tools-temp'
    cmdline_root = ANDROID_SDK_ROOT / 'cmdline-tools'
    latest_dir = cmdline_root / 'latest'

    if not latest_dir.exists():
        if temp_dir.exists():
            shutil.rmtree(temp_dir)
        run(['wget', '-q', '-O', str(zip_path), ANDROID_CMDLINE_TOOLS_URL])
        ensure_dir(temp_dir)
        run(['unzip', '-q', str(zip_path), '-d', str(temp_dir)])
        ensure_dir(cmdline_root)
        extracted_dir = temp_dir / 'cmdline-tools'
        if latest_dir.exists():
            shutil.rmtree(latest_dir)
        shutil.move(str(extracted_dir), str(latest_dir))

    return latest_dir

def install_android_build_tools():
    latest_dir = install_android_commandline_tools()
    sdkmanager = latest_dir / 'bin' / 'sdkmanager'

    env = os.environ.copy()
    env['ANDROID_SDK_ROOT'] = str(ANDROID_SDK_ROOT)
    env['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
    env['PATH'] = f'{latest_dir / "bin"}:{env["PATH"]}'

    run(f'yes | "{sdkmanager}" --sdk_root="{ANDROID_SDK_ROOT}" --licenses >/dev/null', check=False, shell=True, env=env)
    run([
        str(sdkmanager),
        f'--sdk_root={ANDROID_SDK_ROOT}',
        f'build-tools;{ANDROID_BUILD_TOOLS}',
        'platform-tools',
    ], env=env)

    build_tools_dir = ANDROID_SDK_ROOT / 'build-tools' / ANDROID_BUILD_TOOLS
    os.environ['ANDROID_SDK_ROOT'] = str(ANDROID_SDK_ROOT)
    os.environ['PATH'] = f'{build_tools_dir}:{latest_dir / "bin"}:{os.environ["PATH"]}'

def install_dependencies():
    ensure_dir(WORK_ROOT)
    ensure_dir(TOOLS_DIR)

    run(['apt-get', 'update'])
    run([
        'apt-get', 'install', '-y',
        'openjdk-17-jdk-headless',
        'wget',
        'unzip',
        'ca-certificates',
    ])

    install_apktool()
    install_android_build_tools()

    missing = [tool for tool in ('apktool', 'apksigner', 'zipalign', 'keytool') if shutil.which(tool) is None]
    if missing:
        raise RuntimeError(f'Missing required tools after installation: {missing}')

    print('\nInstalled tool paths:')
    for tool in ('apktool', 'apksigner', 'zipalign', 'keytool'):
        print(f'  {tool}: {shutil.which(tool)}')

    print('\nRequested versions:')
    print(f'  apktool: {APKTOOL_VERSION}')
    print(f'  Android build-tools: {ANDROID_BUILD_TOOLS}')

# ── Patch helpers ─────────────────────────────────────────────────────────────
@dataclass
class RuntimeConfig:
    protocol: str
    host: str
    port: str = ''

    @property
    def web_base_url(self) -> str:
        base = f'{self.protocol}://{self.host}'
        if self.port:
            base += f':{self.port}'
        return base

    @property
    def grpc_host(self) -> str:
        return self.host

def require_file(path: Path):
    if not path.is_file():
        raise FileNotFoundError(f'Not found: {path}')

def build_replacements(cfg: RuntimeConfig):
    replacements = [
        ('api.app.nierreincarnation.com', cfg.grpc_host),
        (
            'https://web.app.nierreincarnation.com/assets/release/{0}/database.bin',
            f'{cfg.web_base_url}/assets/release/{{0}}/database.bin',
        ),
        ('https://web.app.nierreincarnation.com', cfg.web_base_url),
        ('https://resources-api.app.nierreincarnation.com/', f'{cfg.web_base_url}/'),
    ]

    for old_value, new_value in replacements:
        if len(new_value.encode('utf-8')) > len(old_value.encode('utf-8')):
            raise ValueError(
                'Replacement too long: '
                f'{old_value!r} -> {new_value!r}. '
                'Use a shorter host name or omit the port.'
            )

    return replacements

def patch_metadata_strings(meta_path: Path, replacements) -> int:
    data = bytearray(meta_path.read_bytes())

    magic = struct.unpack_from('<I', data, 0)[0]
    if magic != METADATA_MAGIC:
        raise RuntimeError(f'Bad metadata magic: 0x{magic:08X}')

    version = struct.unpack_from('<i', data, 4)[0]
    print(f'  metadata v{version}, {len(data)} bytes')

    sl_off, sl_size = struct.unpack_from('<II', data, HDR_STRING_LITERAL_OFF)
    sld_off, sld_size = struct.unpack_from('<II', data, HDR_STRING_LITERAL_DATA_OFF)
    n_entries = sl_size // 8
    print(f'  stringLiteral: {n_entries} entries @ 0x{sl_off:X}')
    print(f'  stringLiteralData: {sld_size} bytes @ 0x{sld_off:X}')

    patched = 0
    for old_str, new_str in replacements:
        old_bytes = old_str.encode('utf-8')
        new_bytes = new_str.encode('utf-8')

        blob_pos = data.find(old_bytes, sld_off, sld_off + sld_size)
        if blob_pos < 0:
            print(f'  [!] NOT FOUND in blob: {old_str!r}')
            continue

        data_index = blob_pos - sld_off
        entry_found = False
        for i in range(n_entries):
            entry_offset = sl_off + i * 8
            entry_len, entry_index = struct.unpack_from('<II', data, entry_offset)
            if entry_index == data_index and entry_len == len(old_bytes):
                struct.pack_into('<I', data, entry_offset, len(new_bytes))
                entry_found = True
                print(f'  entry #{i}: length {entry_len} -> {len(new_bytes)}')
                break

        if not entry_found:
            print(f'  [!] No table entry found for {old_str!r} (dataIndex=0x{data_index:X})')
            continue

        data[blob_pos:blob_pos + len(old_bytes)] = new_bytes + (b'\x00' * (len(old_bytes) - len(new_bytes)))
        print(f'  PATCHED: {old_str!r} -> {new_str!r}')
        patched += 1

    meta_path.write_bytes(data)
    return patched

def patch_manifest(manifest_path: Path) -> bool:
    text = manifest_path.read_text(encoding='utf-8')

    if 'networkSecurityConfig' in text:
        print('  networkSecurityConfig already present')
        return True

    new_attr = 'android:networkSecurityConfig="@xml/network_security_config"'
    patched_text, count = re.subn(r'<application\b', f'<application {new_attr}', text, count=1)
    if count != 1:
        raise RuntimeError('Could not find <application> tag in AndroidManifest.xml')

    manifest_path.write_text(patched_text, encoding='utf-8')
    print(f'  added {new_attr}')
    return True

NETWORK_SECURITY_CONFIG = '''<?xml version="1.0" encoding="utf-8"?>
<network-security-config>
    <base-config cleartextTrafficPermitted="true" />
</network-security-config>
'''

def create_network_security_config(res_xml_dir: Path) -> bool:
    res_xml_dir.mkdir(parents=True, exist_ok=True)
    output_path = res_xml_dir / 'network_security_config.xml'
    output_path.write_text(NETWORK_SECURITY_CONFIG, encoding='utf-8')
    print(f'  wrote {output_path}')
    return True

def patch_libil2cpp(so_path: Path) -> int:
    patched = 0
    with so_path.open('r+b') as handle:
        file_size = handle.seek(0, 2)
        for patch in IL2CPP_PATCHES:
            rva = patch['rva']
            patch_bytes = patch['bytes']
            if rva + len(patch_bytes) > file_size:
                print(f"  [!] SKIP {patch['name']}: RVA 0x{rva:X} beyond file size")
                continue

            handle.seek(rva)
            original = handle.read(len(patch_bytes))
            handle.seek(rva)
            handle.write(patch_bytes)

            patched += 1
            print(f"  {patch['name']} @ 0x{rva:X}: {original.hex()} -> {patch_bytes.hex()}")
            print(f"    {patch['desc']}")

    return patched

def ensure_debug_keystore() -> Path:
    keystore_path = TOOLS_DIR / 'debug.keystore'
    if keystore_path.exists():
        return keystore_path

    run([
        'keytool',
        '-genkeypair',
        '-v',
        '-keystore', str(keystore_path),
        '-storepass', 'android',
        '-alias', 'androiddebugkey',
        '-keypass', 'android',
        '-keyalg', 'RSA',
        '-keysize', '2048',
        '-validity', '10000',
        '-dname', 'CN=Android Debug,O=Android,C=US',
        '-noprompt',
    ])
    return keystore_path

def patch_uploaded_apk(uploaded_apk: Path, cfg: RuntimeConfig) -> Path:
    job_dir = WORK_ROOT / 'job'
    decoded_dir = job_dir / 'decoded'
    unsigned_apk = job_dir / f'{uploaded_apk.stem}-unsigned.apk'
    aligned_apk = job_dir / f'{uploaded_apk.stem}-aligned.apk'
    signed_apk = job_dir / f'{uploaded_apk.stem}-patched.apk'

    if job_dir.exists():
        shutil.rmtree(job_dir)
    job_dir.mkdir(parents=True, exist_ok=True)

    print(f'[*] Decompiling {uploaded_apk.name} ...')
    run(['apktool', 'd', '-f', str(uploaded_apk), '-o', str(decoded_dir)])

    metadata_path = decoded_dir / 'assets/bin/Data/Managed/Metadata/global-metadata.dat'
    so_path = decoded_dir / 'lib/arm64-v8a/libil2cpp.so'
    manifest_path = decoded_dir / 'AndroidManifest.xml'
    res_xml_dir = decoded_dir / 'res/xml'

    for required_path in (metadata_path, so_path, manifest_path):
        require_file(required_path)

    replacements = build_replacements(cfg)

    print(f'\n[*] Patching for server {cfg.host} (web={cfg.web_base_url}, gRPC host={cfg.grpc_host})')

    print('\n[1] Patching global-metadata.dat string literals ...')
    patched_strings = patch_metadata_strings(metadata_path, replacements)
    print(f'    {patched_strings}/{len(replacements)} strings patched')

    print('\n[2] Patching libil2cpp.so ...')
    patched_methods = patch_libil2cpp(so_path)
    print(f'    {patched_methods}/{len(IL2CPP_PATCHES)} methods patched')

    if cfg.protocol == 'http':
        print('\n[3] Enabling cleartext traffic in AndroidManifest.xml ...')
        patch_manifest(manifest_path)

        print('\n[4] Writing network_security_config.xml ...')
        create_network_security_config(res_xml_dir)
    else:
        print('\n[3] HTTPS selected; skipping cleartext network security changes')

    print('\n[5] Rebuilding APK ...')
    run(['apktool', 'b', str(decoded_dir), '-o', str(unsigned_apk)])

    print('\n[6] Aligning APK ...')
    run(['zipalign', '-f', '4', str(unsigned_apk), str(aligned_apk)])

    print('\n[7] Signing APK ...')
    keystore_path = ensure_debug_keystore()
    if signed_apk.exists():
        signed_apk.unlink()
    shutil.copy2(aligned_apk, signed_apk)
    run([
        'apksigner', 'sign',
        '--ks', str(keystore_path),
        '--ks-pass', 'pass:android',
        '--key-pass', 'pass:android',
        '--ks-key-alias', 'androiddebugkey',
        str(signed_apk),
    ])

    print('\n[8] Verifying signature ...')
    run(['apksigner', 'verify', '--verbose', str(signed_apk)])

    print(f'\n[+] Finished: {signed_apk}')
    return signed_apk

# ── Validate config ───────────────────────────────────────────────────────────
server_host = server_host.strip()
server_port = server_port.strip()

if not server_host:
    raise ValueError('server_host cannot be empty')

if server_port and not server_port.isdigit():
    raise ValueError('server_port must be blank or numeric')

display_port = f':{server_port}' if server_port else ''
print(f'Configured web base URL: {protocol}://{server_host}{display_port}')
print(f'Configured gRPC host patch: {server_host}')

# ── Install dependencies ──────────────────────────────────────────────────────
install_dependencies()

# ── Download, patch, and export APK ──────────────────────────────────────────
apk_url = 'https://archive.org/download/nierreincarnation/Global/apk/NieR%20Re%5Bin%5Dcarnation%203.7.1.apk'
output_name = Path(urllib.parse.unquote(apk_url.split('/')[-1]))
input_apk = WORK_ROOT / output_name.name
ensure_dir(WORK_ROOT)

print(f'Downloading APK from {apk_url} ...')
run(['wget', '-q', '-O', str(input_apk), apk_url])
if not input_apk.exists():
    raise RuntimeError('Download failed')
print(f'Downloaded {input_apk.name} ({input_apk.stat().st_size:,} bytes)')

cfg = RuntimeConfig(protocol=protocol, host=server_host, port=server_port)
output_apk = patch_uploaded_apk(input_apk, cfg)
print(f'Patched APK: {output_apk}')

try:
    from google.colab import files
    files.download(str(output_apk))
except Exception:
    print('Browser download failed; patched APK is at:', output_apk)